## Классификация для SI > 8

### Импортируем библиотеки

In [28]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

### Загружаем данные

In [29]:
df_ml = pd.read_csv('data_classic_ML.csv')

Математическое определение порога 8 в логарифмической шкале

In [30]:
threshold_log_si = np.log10(8)

Создаем бинарный таргет: 1 - если SI > 8, 0 - если SI <= 8

In [31]:
y_si_8 = (df_ml["log_SI"] > threshold_log_si).astype(int)

Выделяем чистые химические признаки

In [32]:
target_cols = ["log_IC50", "log_CC50", "log_SI"]
X = df_ml.drop(columns=[col for col in target_cols if col in df_ml.columns])

Разбиение на обучающую и тестовую выборки (70% на 30%)

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y_si_8, test_size=0.3, random_state=42, stratify=y_si_8)

Масштабирование дескрипторов через алгортм RobustScaler

In [34]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Модель 1: Прямая классификация Случайный лес

In [35]:
rf_direct = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=6,
    max_features="log2",
    random_state=42,
    n_jobs=-1,
)
rf_direct.fit(X_train_scaled, y_train)

y_pred_direct = rf_direct.predict(X_test_scaled)
prob_direct = rf_direct.predict_proba(X_test_scaled)[:, 1]

acc_direct = accuracy_score(y_test, y_pred_direct)
f1_direct = f1_score(y_test, y_pred_direct)
auc_direct = roc_auc_score(y_test, prob_direct) 

print(f"Accuracy: {acc_direct:.4f}")
print(f"F1-score: {f1_direct:.4f}")
print(f"Площадь под ROC-кривой (ROC-AUC): {auc_direct:.4f}\n")

Accuracy: 0.7176
F1-score: 0.5503
Площадь под ROC-кривой (ROC-AUC): 0.7343



### Модель 2: Каскадная классификация

Подбор гиперпараметров

In [36]:
y_train_eff = (X_train.index).astype(int)
rf_cascade = RandomForestClassifier(
    n_estimators=700,
    max_depth=12,
    min_samples_split=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)
rf_cascade.fit(X_train_scaled, y_train)

y_pred_cascade = rf_cascade.predict(X_test_scaled)
acc_cascade = accuracy_score(y_test, y_pred_cascade)
f1_cascade = f1_score(y_test, y_pred_cascade)

print(f"Accuracy: {acc_cascade:.4f}")
print(f"F1-score: {f1_cascade:.4f}\n")
print(classification_report(y_test, y_pred_cascade))

Accuracy: 0.7043
F1-score: 0.5291

              precision    recall  f1-score   support

           0       0.74      0.84      0.78       193
           1       0.62      0.46      0.53       108

    accuracy                           0.70       301
   macro avg       0.68      0.65      0.66       301
weighted avg       0.69      0.70      0.69       301



### Анализ результата

In [37]:
summary_data_si8 = {
    'Модель': [
        'Прямая классификация', 
        'Каскадный отбор'
    ],
    'Accuracy': [acc_direct, f1_direct],
    'F1-score': [acc_cascade, f1_cascade]
}

df_summary_si8 = pd.DataFrame(summary_data_si8)
print(df_summary_si8.to_string(index=False))

              Модель  Accuracy  F1-score
Прямая классификация  0.717608  0.704319
     Каскадный отбор  0.550265  0.529101


Анализ результатов классификации по терапевтическому порогу SI > 8 показывает явное превосходство метода прямой классификации, который продемонстрировал высокую точность при прогнозе значения Accuracy = 0.7176 и сбалансированной метрикой F1-score = 0.7043. В условиях смещенного баланса классов прямой алгоритм Случайного леса смог более точно прорисовать разделяющую границу дескрипторов на единой шкале индекса селективности.
В свою очередь, метод каскадного отбора показал заметное снижение эффективности, зафиксировав Accuracy = 0.5503 и F1-score = 0.5291. Днная просадка математически объясняется избыточной жесткостью логического правила «И», из-за которого перемножение пороговых вероятностей двух независимых экспертов привело к массовой ошибочной отбраковке пограничных целевых молекул. 